# 1) Import libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import seaborn as sns

# 2) Load Dataset

In [8]:
df=pd.read_excel("/workspaces/Customer_Segmentation_KMeans_Clustering/data/Online Retail.xlsx")
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [52]:
df[df["CustomerID"] == 12346]

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
61619,541431,23166,MEDIUM CERAMIC TOP STORAGE JAR,74215,2011-01-18 10:01:00,1.04,12346.0,United Kingdom
61624,C541433,23166,MEDIUM CERAMIC TOP STORAGE JAR,-74215,2011-01-18 10:17:00,1.04,12346.0,United Kingdom


# 3) Inspect and Clean

In [28]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[us]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 33.1+ MB


In [ ]:
df.InvoiceNo.value_counts()
print("Number of Cancelled invoices:", df.InvoiceNo.str.startswith('C').sum())
print("Number of negative entries in the 'Quantity' column:", (df['Quantity'] < 0).sum())
print("Number of negative entries in the 'UnitPrice' column:", (df['UnitPrice'] < 0).sum())


Number of Cancelled invoices: 9288
Number of negative entries in the 'Quantity' column: 10624
Number of negative entries in the 'UnitPrice' column: 2
531285
10624


np.float64(1.0)

In [36]:
print("Number of duplicates:", df.duplicated().sum())
df.isna().sum()/len(df)*100

Number of duplicates: 5268


InvoiceNo       0.000000
StockCode       0.000000
Description     0.268311
Quantity        0.000000
InvoiceDate     0.000000
UnitPrice       0.000000
CustomerID     24.926694
Country         0.000000
dtype: float64

In [44]:
df["Country"].value_counts()

Country
United Kingdom          495478
Germany                   9495
France                    8557
EIRE                      8196
Spain                     2533
Netherlands               2371
Belgium                   2069
Switzerland               2002
Portugal                  1519
Australia                 1259
Norway                    1086
Italy                      803
Channel Islands            758
Finland                    695
Cyprus                     622
Sweden                     462
Unspecified                446
Austria                    401
Denmark                    389
Japan                      358
Poland                     341
Israel                     297
USA                        291
Hong Kong                  288
Singapore                  229
Iceland                    182
Canada                     151
Greece                     146
Malta                      127
United Arab Emirates        68
European Community          61
RSA                         58


In [3]:
def wrangle(df):
    """
    Cleans and transforms the Online Retail dataset into a
    customer-level dataset suitable for K-Means clustering.

    Parameters
    ----------
    df : pandas.DataFrame
        Raw transactional data.

    Returns
    -------
    pandas.DataFrame
        Customer-level dataframe with engineered features.
    """

    # Create a copy to avoid modifying the original dataframe
    df = df.copy()

    # Remove duplicate rows
    df.drop_duplicates(inplace=True)

    # Remove rows with missing values
    df.dropna(inplace=True)

    # Remove transactions that were completely cancelled
    # (Remove both the original purchase and its credit note)
    

    # Separate purchases and cancellations
    purchases = df[df["Quantity"] > 0].copy()
    cancellations = df[df["Quantity"] < 0].copy()

    # Create absolute quantity for matching
    purchases["AbsQuantity"] = purchases["Quantity"]
    cancellations["AbsQuantity"] = cancellations["Quantity"].abs()

    # Match purchases with cancellations
    matched = purchases.merge(
        cancellations,
        on=["CustomerID", "StockCode", "UnitPrice", "AbsQuantity"],
        suffixes=("_purchase", "_cancel")
    )

    # Get all invoice numbers involved in complete cancellations
    invoices_to_remove = (
        set(matched["InvoiceNo_purchase"])
        .union(set(matched["InvoiceNo_cancel"]))
    )

    # Remove matched purchase/cancellation pairs
    df = df[~df["InvoiceNo"].isin(invoices_to_remove)]

    # Remove any remaining negative quantities or invalid prices
    df = df[(df["Quantity"] > 0) & (df["UnitPrice"] > 0)]

    # Create TotalPrice feature
    df["TotalPrice"] = df["Quantity"] * df["UnitPrice"]

    # Reference date for Recency calculation
    reference_date = df["InvoiceDate"].max()

    # Aggregate transaction-level data to customer level
    customer_df = (
        df.groupby("CustomerID")
          .agg(
              TotalSpent=("TotalPrice", "sum"),
              Orders=("InvoiceNo", "nunique"),
              ItemsPurchased=("Quantity", "sum"),
              CountriesPurchased=("Country", "nunique"),
              LastPurchase=("InvoiceDate", "max")
          )
          .reset_index()
    )

    # Feature Engineering

    # Average spend per order
    customer_df["AvgOrderValue"] = (
        customer_df["TotalSpent"] / customer_df["Orders"]
    )

    # Days since last purchase
    customer_df["Recency"] = (
        reference_date - customer_df["LastPurchase"]
    ).dt.days

    # Drop helper column
    customer_df.drop(columns="LastPurchase", inplace=True)

    return customer_df

In [9]:
customer_df=wrangle(df)
customer_df.head()



,CustomerID,TotalSpent,Orders,ItemsPurchased,CountriesPurchased,AvgOrderValue,Recency
0,12347.0,4310.00,7,2458,1,615.714286,1
1,12348.0,1797.24,4,2341,1,449.310000,74
2,12349.0,1757.55,1,631,1,1757.550000,18
3,12350.0,334.40,1,197,1,334.400000,309
4,12352.0,1240.73,4,380,1,310.182500,35


In [10]:
# Final clean dataset
customer_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4239 entries, 0 to 4238
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   CustomerID          4239 non-null   float64
 1   TotalSpent          4239 non-null   float64
 2   Orders              4239 non-null   int64  
 3   ItemsPurchased      4239 non-null   int64  
 4   CountriesPurchased  4239 non-null   int64  
 5   AvgOrderValue       4239 non-null   float64
 6   Recency             4239 non-null   int64  
dtypes: float64(3), int64(4)
memory usage: 231.9 KB


In [11]:
# Save prepared dataframe
customer_df.to_csv("clean_customer_data.csv", index=False)

## Data cleaning report

To ensure the quality and reliability of the customer segmentation analysis, several preprocessing steps were performed on the raw transactional dataset before applying K-Means clustering.
### 1. Removed Duplicate Records
Duplicate transactions were identified and removed to prevent the same purchase from being counted multiple times. Retaining duplicate records would artificially inflate customer spending, purchase frequency, and total units purchased.
### 2. Removed Missing Values
Rows containing missing values were removed from the dataset. In particular, transactions without a `CustomerID` could not be attributed to any customer and therefore could not contribute to customer-level feature engineering.
### 3. Identified and Removed Fully Cancelled Transactions
The dataset contains cancellation records (credit notes), typically represented by negative quantities and invoice numbers beginning with the letter **"C"**.
Instead of simply removing cancellation records, an additional matching process was performed to identify transactions that had been completely reversed.

Purchases and cancellations were matched using the following attributes:

* CustomerID
* StockCode
* UnitPrice
* Absolute Quantity

When an exact purchase-cancellation pair was identified, **both the original purchase and its corresponding cancellation were removed**. This prevented cancelled purchases from inflating customer spending and purchase behaviour.

### 4. Removed Remaining Invalid Transactions

After removing matched cancellation pairs, any remaining transactions with:

* Negative quantities
* Non-positive unit prices

were removed since they do not represent valid completed sales.

### Feature Engineering

Because K-Means clusters observations rather than transactions, the dataset was transformed from **transaction-level** data into **customer-level** data.

Each customer became a single observation with aggregated behavioural features.

##### Total Spending (`TotalSpent`)

The total monetary value of purchases made by each customer.

This was calculated by first computing:

> **TotalPrice = Quantity × UnitPrice**

and then summing `TotalPrice` across all transactions belonging to each customer.

---

##### Number of Orders (`Orders`)

The number of unique invoices associated with each customer.

This represents how frequently a customer placed orders during the observation period.

---

##### Items Purchased (`ItemsPurchased`)

The total quantity of products purchased by each customer across all completed transactions.

This measures the purchasing volume of each customer.

---

##### Countries Purchased (`CountriesPurchased`)

The number of unique countries from which each customer placed orders.

Although this feature was engineered, exploratory analysis showed that most customers purchased from only one country, making this variable potentially less informative for clustering.

---

##### Average Order Value (`AvgOrderValue`)

The average monetary value of each customer's orders.

It was calculated as:

**Average Order Value = TotalSpent / Orders**

This feature helps distinguish customers who make frequent low-value purchases from those who place fewer but higher-value orders.

---

##### Recency (`Recency`)

Recency measures the number of days since a customer's last purchase.

It was calculated as the difference between:

* the most recent transaction date in the dataset (reference date), and
* each customer's most recent purchase date.

Customers with lower recency values have purchased more recently and are considered more active than customers with higher recency values.

---

#### Transformation

After feature engineering, the dataset was transformed from over **541,000 transaction records** into a customer-level dataset containing one row per customer.

This transformed dataset served as the input for customer segmentation using the K-Means clustering algorithm.

---

#### Assumptions

The following assumptions were made during data preparation:

1. **Duplicate records** were assumed to be accidental duplicates rather than legitimate repeated transactions.

2. Transactions with **missing Customer IDs** could not be associated with any customer and were therefore excluded from the analysis.

3. Purchase and cancellation records sharing the same **CustomerID**, **StockCode**, **UnitPrice**, and **absolute Quantity** were assumed to represent complete reversals of the same transaction and were removed from the dataset.

4. Remaining transactions with **negative quantities** or **non-positive unit prices** were assumed not to represent valid completed purchases.

5. Customer behaviour was represented using aggregated purchasing statistics, meaning that the order in which purchases occurred was not considered during clustering.

6. The latest transaction date in the dataset was assumed to represent the end of the observation period and was used as the reference point for calculating customer recency.

7. The engineered customer-level features were assumed to adequately capture purchasing behaviour for the purpose of customer segmentation using K-Means clustering.
